In [6]:
import pandas as pd
from pathlib import Path
f = Path('../data/processed/05_features_full.parquet')
print('exists:', f.exists(), '| size MB:', round(f.stat().st_size/1e6, 1) if f.exists() else '-')
d = pd.read_parquet(f, columns=['Timestamp', 'split', 'Is Laundering'])
print(d.shape, d['Timestamp'].dtype)
print(d['split'].value_counts())

exists: True | size MB: 283.8
(5078345, 3) datetime64[ns]
split
train    3554957
val       761749
test      761639
Name: count, dtype: int64


In [7]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED)

DATA_PROC = Path('../data/processed')
OUT_DIR   = Path('../outputs')
(OUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'tables').mkdir(parents=True, exist_ok=True)

print('xgboost', xgb.__version__)  # expect 2.1.4

xgboost 2.1.4


In [4]:
df = pd.read_parquet(DATA_PROC / '05_features_full.parquet')

print(f'rows: {len(df):,}  cols: {df.shape[1]}') 
print(df['split'].value_counts())
assert len(df) == 5_078_345, 'row count does not match the registered artefact'

rows: 5,078,345  cols: 28
split
train    3554957
val       761749
test      761639
Name: count, dtype: int64


In [5]:
day_counts = df['Timestamp'].dt.date.value_counts().sort_index()
M = day_counts.median()
thresh = 0.05 * M
wind_days = set(day_counts[day_counts < thresh].index)

df['is_wind_down'] = df['Timestamp'].dt.date.isin(wind_days)

test_mask = df['split'] == 'test'
main_mask = test_mask & ~df['is_wind_down']
wind_mask = test_mask & df['is_wind_down']

print(f'median M = {M:,.0f}  threshold = {thresh:,.1f}')
print(f'wind-down days: {sorted(wind_days)}')
print(f'test-main  {main_mask.sum():,} rows / {df.loc[main_mask, "Is Laundering"].sum():,} illicit')
print(f'wind-down  {wind_mask.sum():,} rows / {df.loc[wind_mask, "Is Laundering"].sum():,} illicit')

# Registered constants — fail rather than drift
assert M == 207_406, f'median changed: {M}'
assert main_mask.sum() == 760_531 and wind_mask.sum() == 1_108
assert df.loc[main_mask, 'Is Laundering'].sum() == 906
assert df.loc[wind_mask, 'Is Laundering'].sum() == 655

median M = 207,406  threshold = 10,370.3
wind-down days: [datetime.date(2022, 9, 11), datetime.date(2022, 9, 12), datetime.date(2022, 9, 13), datetime.date(2022, 9, 14), datetime.date(2022, 9, 15), datetime.date(2022, 9, 16), datetime.date(2022, 9, 17), datetime.date(2022, 9, 18)]
test-main  760,531 rows / 906 illicit
wind-down  1,108 rows / 655 illicit


In [8]:
df['hour']        = df['Timestamp'].dt.hour
df['day_of_week'] = df['Timestamp'].dt.dayofweek
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# XGBoost only reads numbers, so text columns become integer codes.
# Notebook 07 must redo this the same way or the two models won't be comparable.
CAT_COLS = ['Receiving Currency', 'Payment Currency', 'Payment Format']
for c in CAT_COLS:
    df[c] = df[c].astype('category').cat.codes

TAB_FEATS = ['Amount Received', 'Amount Paid', 'From Bank', 'To Bank',
             'Receiving Currency', 'Payment Currency', 'Payment Format',
             'hour', 'day_of_week', 'is_weekend']

print(f'M1 uses {len(TAB_FEATS)} features')
json.dump(TAB_FEATS, open(OUT_DIR / 'models' / '06_m1_features.json', 'w'), indent=2)

M1 uses 10 features


In [9]:
# Save the code→category mapping so Notebook 07 can assert it matches.
maps = {c: dict(enumerate(pd.read_parquet(DATA_PROC / '05_features_full.parquet',
                                          columns=[c])[c].astype('category').cat.categories))
        for c in CAT_COLS}
json.dump(maps, open(OUT_DIR / 'models' / '06_category_maps.json', 'w'), indent=2)
for c, m in maps.items():
    print(c, '→', len(m), 'categories')

Receiving Currency → 15 categories
Payment Currency → 15 categories
Payment Format → 7 categories


In [ ]:
def xy(split):
    m = df['split'] == split
    return df.loc[m, TAB_FEATS], df.loc[m, 'Is Laundering']

X_train, y_train = xy('train')
X_val,   y_val   = xy('val')
X_main,  y_main  = df.loc[main_mask, TAB_FEATS], df.loc[main_mask, 'Is Laundering']
X_test,  y_test  = df.loc[test_mask, TAB_FEATS], df.loc[test_mask, 'Is Laundering']

# Roughly 1 illicit in every 1,244 training rows. Without this weight the model
# learns to call everything clean and still looks 99.9% accurate.
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'scale_pos_weight = {scale_pos:.1f}')  

scale_pos_weight = 1243.7


In [ ]:
# ref=dtrain forces val and test to reuse the training bin edges, so all three
# matrices measure the same thing. It also keeps memory flat on 3.5M rows.
dtrain = xgb.QuantileDMatrix(X_train, label=y_train)
dval   = xgb.QuantileDMatrix(X_val,  label=y_val,  ref=dtrain)
dmain  = xgb.QuantileDMatrix(X_main, label=y_main, ref=dtrain)
dtest  = xgb.QuantileDMatrix(X_test, label=y_test, ref=dtrain)

# Every value here is copied verbatim into Notebook 07. Change one, change both.
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'aucpr',        # matches the H1 primary metric
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos,
    'max_delta_step': 1,           # steadies the updates under that huge weight
    'tree_method': 'hist',
    'device': 'cpu',
    'seed': SEED,
}

booster = xgb.train(
    params, dtrain, num_boost_round=500,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,      
    verbose_eval=50,
)
print(f'best iteration: {booster.best_iteration}')

[0]	train-aucpr:0.00803	val-aucpr:0.00940
[50]	train-aucpr:0.03270	val-aucpr:0.02314
[100]	train-aucpr:0.03958	val-aucpr:0.03014
[150]	train-aucpr:0.05185	val-aucpr:0.03617
[200]	train-aucpr:0.06348	val-aucpr:0.04073
[250]	train-aucpr:0.07026	val-aucpr:0.04381
[300]	train-aucpr:0.07858	val-aucpr:0.04581
[350]	train-aucpr:0.08359	val-aucpr:0.04736
[400]	train-aucpr:0.09101	val-aucpr:0.04861
[450]	train-aucpr:0.09570	val-aucpr:0.05088
[499]	train-aucpr:0.10083	val-aucpr:0.05157
best iteration: 481


In [13]:
# Freeze the classification threshold on VALIDATION only (registered decision D4).
# iteration_range is explicit so M1 and M2 are scored on the same rule, not a library default.
p_val = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))

# 400 candidate cut-offs from the top 10% of scores. Below that nothing is flagged
# anyway, since positives are 0.08% of rows.
grid = np.unique(np.quantile(p_val, np.linspace(0.90, 0.99999, 400)))
f1s  = [f1_score(y_val, (p_val >= t).astype(int), zero_division=0) for t in grid]
best_i = int(np.argmax(f1s))

# If the best F1 sits on the low edge of the grid, the true optimum may lie below it.
assert best_i > 0, 'optimum at grid floor — widen the quantile range below 0.90'

THRESH = float(grid[best_i])
N_TREES = int(booster.best_iteration) + 1   # trees actually used; carried downstream
print(f'frozen threshold = {THRESH:.6f}   val minority-F1 = {f1s[best_i]:.4f}   trees = {N_TREES}')

json.dump({'threshold': THRESH, 'val_f1': float(f1s[best_i]),
           'chosen_on': 'validation', 'n_trees_used': N_TREES,
           'num_boost_round': 500, 'early_stopping_rounds': 50},
          open(OUT_DIR / 'models' / '06_m1_threshold.json', 'w'), indent=2)

frozen threshold = 0.957998   val minority-F1 = 0.1207   trees = 482


In [ ]:
# The model is now an Arguement , not captured from the notebook scope. This is what
# lets the 2000-round supplementary fit be scored by the same function without
# silently picking up the wrong booster.
def evaluate(bst, dm, y, label, thresh):
    p = bst.predict(dm, iteration_range=(0, bst.best_iteration + 1))
    yhat = (p >= thresh).astype(int)
    return {
        'population': label,
        'n': int(len(y)),
        'illicit': int(y.sum()),
        'pr_auc': float(average_precision_score(y, p)),
        'f1_minority': float(f1_score(y, yhat, zero_division=0)),
        'roc_auc': float(roc_auc_score(y, p)),   # reported for comparability only
    }, p

m_main, p_main = evaluate(booster, dmain, y_main, 'test-main', THRESH)
m_full, p_test = evaluate(booster, dtest, y_test, 'test-full', THRESH)
results = pd.DataFrame([m_main, m_full])
print(results.to_string(index=False))

# A near-perfect score on this problem almost always means something leaked.
if m_main['pr_auc'] > 0.90:
    print('\nSTOP — PR-AUC above 0.90. Run the label-permutation test before continuing.')

population      n  illicit   pr_auc  f1_minority  roc_auc
 test-main 760531      906 0.085767     0.127856 0.964851
 test-full 761639     1561 0.181600     0.228654 0.973581


In [15]:
booster.save_model(OUT_DIR / 'models' / 'm1_baseline.json')

# Saved With the row index attached, plus an is_main flag so Notebook 07 never has to
# re-derive the wind-down split. Notebook 07 lines M1 and M2 up by index.
pd.DataFrame({'m1_prob': p_test,
              'is_main': main_mask.loc[df.index[test_mask]].values},   # <-- your cell-3 mask name
             index=df.index[test_mask]) \
  .to_parquet(OUT_DIR / 'models' / 'm1_test_probs.parquet')

np.save(OUT_DIR / 'models' / 'm1_test_probs.npy', p_test)  # positional backup only
results.to_csv(OUT_DIR / 'tables' / '06_m1_metrics.csv', index=False)

assert m_main['n'] == 760_531 and m_full['n'] == 761_639, 'population sizes off-spec'
print(f'saved 4 files · trees used {N_TREES} · threshold {THRESH:.6f}')

saved 4 files · trees used 482 · threshold 0.957998


In [16]:
# Confirm the saved is_main flag reproduces the registered test-main population.
chk = pd.read_parquet(OUT_DIR / 'models' / 'm1_test_probs.parquet')
print(f"rows {len(chk)} · is_main True {int(chk['is_main'].sum())} · index unique {chk.index.is_unique}")

rows 761639 · is_main True 760531 · index unique True
